In [1]:
#%pip install subprocess
import uproot
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import subprocess
from enum import IntEnum, auto
from pathlib import Path

In [2]:
class Setting(IntEnum):
    g_nReal = auto()
    g_nZ = auto()
    g_nAMass = auto()
    g_nConEBin = auto()
    alpha_parameter = auto()
    sigma_parameter = auto()
    fine_binning = auto()
    rough_binning = auto()
    E_steps = auto()

setting_definer = {
    Setting.g_nReal : "const int g_nReal = ",
    Setting.g_nZ : "const int g_nZ = ",
    Setting.g_nAMass : "const int g_nAMass = ",
    Setting.g_nConEBin : "const int g_nConEBin = "
}

value_setting = {Setting.g_nReal,
                 Setting.g_nZ,
                 Setting.g_nAMass,
                 Setting.g_nConEBin}

std_path = Path(r"C:\RAINIER\sample_folder\fluctuation_analysis")

In [8]:
def get_interval(energy, myData, lower, upper):
        intervalled_data = []
        for k in range(myData.size):
            if energy[k] >= lower and energy[k] <= upper:
                intervalled_data.append(myData[k])
        intervalled_data = np.array(intervalled_data)
        return intervalled_data

def autocorr(x, energy, full_data, lower, upper):
    h1 = get_interval(energy, full_data, lower, upper)
    h2 = get_interval(energy, full_data, lower+x, upper+x)
    c1 = 0
    c2 = 0
    while h1.size > h2.size:
        c1 += 1
        h1 = h1[:-1]
    while h1.size < h2.size:
        c2 += 1
        h2 = h2[:-1]
    return np.mean(h1*h2)/(np.mean(h1)*np.mean(h2))

def lvl_dens(E, E_step, energy, full_data, sigma = settings[Setting.sigma_parameter], alpha = settings[Setting.alpha_parameter]):
        return 1/ ((autocorr(0, energy, full_data, E, E + E_step)-1)*2*sigma*np.sqrt(np.pi)/alpha)

def get_nld(spin_array, s):
    return spin_array[s]

def get_smooth(nld):
    pd_data = pd.Series(nld)
    fine = pd_data.rolling(window=settings[Setting.fine_binning], center=True).mean()
    rough = pd_data.rolling(window=settings[Setting.rough_binning], center=True).mean()
    return fine, rough

def plot_nld(energy, nld, save_path, file_name = "myNLD.png"):
    plt.figure()
    plt.plot(energy, nld)
    plt.xlabel("E in MeV")
    plt.ylabel("Levels in Energy Bin")
    plt.title("NLD für alle spins")
    plt.yscale("log")
    plt.savefig(save_path / "NLD_input" / file_name, dpi=300)
    plt.close()

def plot_smooth(energy, nld, fine, rough, save_path, file_name = "mySmooth.png"):
    plt.figure()
    plt.plot(energy, nld)
    plt.plot(energy, rough)
    plt.plot(energy, fine)
    plt.xlabel("E in MeV")
    plt.ylabel("Levels in Energy Bin")
    plt.title("Rough/fine smoothing")
    plt.yscale("log")
    plt.savefig(save_path / "Smoothing" / file_name, dpi=300)
    plt.close()

def plot_stationary(energy, d_full, save_path, file_name):
    plt.figure()
    plt.plot(energy, d_full)
    plt.savefig(save_path / "Stationary" / file_name, dpi=300)
    plt.close()

def plot_autocorrelation(eps_start, eps_end, eps_step, energy, full_data, save_path, file_name, *, interval_low = 5, interval_high = 6):
    epsilons = np.arange(eps_start,eps_end,eps_step)
    plt.figure()
    plt.title("Autocorrelation function for 5MeV-6MeV")
    plt.plot(epsilons, [autocorr(eps, energy, full_data, interval_low, interval_high) for eps in epsilons])
    plt.xlim(0,0.5) 
    plt.savefig(save_path / "Autocorrelation" / file_name, dpi=300)
    plt.close()

def get_fa_density(E_start, E_end, E_step, energy, full_data):
    E_int = np.arange(E_start, E_end, E_step)
    n = len(E_int)
    result = np.zeros((2,n))
    for k in range(n):
        result[0][k] = E_int[k]
        result[1][k] = lvl_dens(E_int[k], E_step, energy, full_data)
    return result

def plot_comparison(E_int, fa_dens, energy, nld, save_path, file_name):
    plt.figure()
    plt.plot(energy, nld)
    plt.scatter(E_int, fa_dens, color = "orange")
    plt.yscale("log")
    plt.title("Fluctuation Analysis compared to original level scheme")
    plt.xlabel("E in MeV")
    plt.ylabel("Levels per MeV")
    plt.savefig(save_path / "Comparison" / file_name, dpi=300)
    plt.close()

def plot_fluctuation_analysis_spin(result, s, save_path, *, print_nld = True, print_smooth = True, print_stationary = True, print_autocorrelation = True, print_comparison = True):
    fa_folder = save_path
    fa_folder.mkdir(exist_ok=True)
    (fa_folder/"NLD_input").mkdir(exist_ok=True)
    (fa_folder/"Smoothing").mkdir(exist_ok=True)
    (fa_folder/"Stationary").mkdir(exist_ok=True)
    (fa_folder/"Autocorrelation").mkdir(exist_ok=True)
    (fa_folder/"Comparison").mkdir(exist_ok=True)
    file_name = "spin"+str(s)
    energy = result["energy"]
    nld = result[str(s)]["nld"]
    fine = result[str(s)]["fine"]
    rough = result[str(s)]["rough"]
    stationary = result[str(s)]["stationary"]
    E_int = result[str(s)]["fin_energy_int"]
    fa_dens = result[str(s)]["fa_dens"]
    if print_nld:
        plot_nld(energy, nld, save_path, file_name)
    if print_smooth:
        plot_smooth(energy, nld, fine, rough, save_path, file_name)
    if print_stationary:
        plot_stationary(energy, stationary, save_path, file_name)
    if print_autocorrelation:
        plot_autocorrelation(0, 0.5, 0.001, energy, stationary, save_path, file_name)
    if print_comparison:
        plot_comparison(E_int, fa_dens, energy, nld, save_path, file_name)

def plot_fluctuation_analysis(result, save_path, *, print_nld = True, print_smooth = True, print_stationary = True, print_autocorrelation = True, print_comparison = True):
    for key in result.keys():
        if key != "energy":
            plot_fluctuation_analysis_spin(result, key, save_path, print_nld=print_nld, print_smooth=print_smooth, print_stationary=print_stationary, print_autocorrelation=print_autocorrelation, print_comparison=print_comparison)

def fluctuation_analysis_spin(s, energy, spin_array):
    nld = get_nld(spin_array, s)
    fine, rough = get_smooth(nld)
    d_full = fine/rough
    E_int, fa_dens = get_fa_density(0, 7, settings[Setting.E_steps], energy, d_full)
    return {
        "nld" : nld,
        "fine" : fine,
        "rough" : rough,
        "stationary" : d_full,
        "fin_energy_int" : E_int,
        "fa_dens" : fa_dens
    }

def run_simulation():
    apply_settings()
    return subprocess.run(["cmd", "/c", "root", r"C:\RAINIER\RAINIER.C"], capture_output=True, text=True, cwd=r"C:\RAINIER\sample_folder").stdout

def get_level_data(run):
    cut = run.find("More levels exist at higher spins")
    #print(run[cut:])
    cut2 = run[cut:].find("E(MeV)")
    first = run[cut+cut2+10:]
    cut3 = first.find("Total Number of Levels")
    second = first[:cut3]
    return second

def get_level_arrays(text):
    energy = []
    spin_array = []
    for k in range(len(text)):
        ch = text[k]
        if ch != '.':
            continue
        a = 1
        while text[k-(a+1)].isdigit():
            a = a+1
        num = float(text[k-a:k+4])
        energy.append(num)
        l = 20
        b = 8
        spin_array_one_energy = [[] for x in range(20)]
        while l > 0:
            if text[k+b] == " ":
                b = b+1
            c = 1
            #print("b:",b)
            while text[k+b+c] != "|":
                c = c+1
            #print("c:",c)
            spin_val = text[k+b:k+b+c]
            #print(spin_val)
            spin_val = spin_val.replace("\n", "")
            spin_array_one_energy[l-1] = int(spin_val)
            #print("spin"+str(20-l)+":"+spin_val)
            #print("Spin"+str(19-l)+":"+text[k+b:k+b+c+2])
            b = b+c+1
            l = l-1
        #print(spin_array_one_energy)
        spin_array.append(spin_array_one_energy)
    energy = np.array(energy)
    spin_array = np.array(spin_array)
    spin_array = spin_array.T
    return (energy,spin_array)

def fluctuation_analysis(run):
    energy, spin_array = get_level_arrays(get_level_data(run))
    result = {"energy" : energy}
    for s in range(spin_array.shape[0]):
        result[str(s)] = fluctuation_analysis_spin(s, energy, spin_array)
    return result

def replace_val(text, definer, new_val):
    pos = text.find(definer)
    start_val = pos + len(definer)
    end_val = start_val
    while text[end_val] != ";":
        end_val = end_val + 1
    val = text[start_val:end_val]
    if val != str(new_val):
        print("(changed) "+text[pos:start_val] + str(new_val))
        return text[:start_val] + str(new_val) + text[end_val:]
    else:
        print(text[pos:end_val])
        return text

def apply_settings():
    with open(r"C:\RAINIER\sample_folder\settings.h", "r") as f:
        text = f.read()
    with open(r"C:\RAINIER\sample_folder\settings.h", "w") as f:
        for key in settings.keys():
            if key in value_setting:
                text = replace_val(text, setting_definer[key], settings[key])
        f.write(text)

In [4]:
settings = {
    Setting.g_nReal : 1,
    Setting.g_nZ : 60,
    Setting.g_nAMass : 144,
    Setting.g_nConEBin : 600,
    Setting.alpha_parameter : 0.45,
    Setting.sigma_parameter : 5,
    Setting.fine_binning : 8,
    Setting.rough_binning : 12,
    Setting.E_steps : 0.5
}

In [6]:
run = run_simulation()

const int g_nReal = 1
const int g_nZ = 60
const int g_nAMass = 144
const int g_nConEBin = 600


In [7]:
fa = fluctuation_analysis(run)
plot_fluctuation_analysis(fa, std_path)

In [ ]:
a = 1